# 4_Metadata


1. Add Dataset level metadata
2. Add Sample level metadata
3. Add ground truth labels

In [1]:
import shutil
from pathlib import Path

import pandas as pd
import h5py
from anndata.io import read_elem, write_elem
from dotenv import dotenv_values


In [2]:
# Project-wide Constants
CONFIG = dotenv_values("config.env") # assuming kernel located at the root of the directory
DATA_ROOT_DIR = Path(CONFIG['DATADIR'])
RCPARAMS = CONFIG['MPLSTYLE']

In [3]:
# Notebook-specific constants; automatically derived depending on user input above
DATADIR = DATA_ROOT_DIR / "2b_clean"

DATASET_METADATA = 'resources/dataset_metadata.csv' # may have to change based on kernel location
SAMPLE_METADATA = 'resources/sample_metadata.csv'
#KNOWN_LABELS=r"/REDACTED-path/HypoMap3D/known_labels.csv" 
KNOWN_LABELS= DATADIR / "known_labels.csv" 

concat_fp = Path(DATADIR, f'concat.h5ad')
annconcat_fp = Path(DATADIR, f'annconcat.h5ad')


In [4]:
shutil.copy2(concat_fp, annconcat_fp) #copy. ~20 min

PosixPath('/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/annconcat.h5ad')

### Let's lazily load the obs from h5ad file

In [5]:
h5ad = h5py.File(annconcat_fp, 'r+')

obs_df = read_elem(h5ad['obs'])

obs_df = obs_df.loc[:, [c for c in obs_df.columns if c != "manually_removed"]] #don't need this column 

obs_df["Sample"] = obs_df["Dataset"].astype(str) + "_" + obs_df["Sample"].astype(str)

obs_df.head()

,CellID,Dataset,Sample,total_counts,n_genes,pct_mito
SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01,SC1-01_SAMN18754353,17459.0,5884,0.131737
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01,SC1-01_SAMN18754353,17067.0,5727,0.117185
SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01,SC1-01_SAMN18754353,5183.0,2578,0.501640
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01,SC1-01_SAMN18754353,11905.0,4965,0.201596
SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01,SC1-01_SAMN18754353,2001.0,1125,2.598701


### Let's add the Dataset Level Metadata

In [6]:
dataset_metadata_df = pd.read_csv(DATASET_METADATA, index_col=0, dtype='category')

In [7]:
dataset_metadata_df.head()

,Author,Source,Data_source_link,GEO,Publication_Link,Suspension,Technology
ReferenceID,,,,,,,
SC1-01,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
SC1-02,Anderson,SRA,NaN,GSE193921,https://doi.org/10.1016/j.neuron.2021.12.002,scRNAseq,10x v3
SC1-03,Campbell,SRA,NaN,GSE93374,https://www.ncbi.nlm.nih.gov/pubmed/28166221,scRNAseq,Drop-seq
SC1-04,Chen,SRA,NaN,GSE87544,https://www.ncbi.nlm.nih.gov/pubmed/28355573,scRNAseq,Drop-seq
SC1-05,Dowsett,SRA,NaN,GSE207736,https://www.nature.com/articles/s42255-022-006...,snRNAseq,10x v3


In [8]:

obs_df = pd.merge(obs_df, dataset_metadata_df, left_on="Dataset", right_index=True, how="left")

In [9]:
obs_df.head()

,CellID,Dataset,Sample,total_counts,n_genes,pct_mito,Author,Source,Data_source_link,GEO,Publication_Link,Suspension,Technology
SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01,SC1-01_SAMN18754353,17459.0,5884,0.131737,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01,SC1-01_SAMN18754353,17067.0,5727,0.117185,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01,SC1-01_SAMN18754353,5183.0,2578,0.501640,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01,SC1-01_SAMN18754353,11905.0,4965,0.201596,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01,SC1-01_SAMN18754353,2001.0,1125,2.598701,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3


### Let's add the Sample Level Metadata

In [10]:
sample_metadata_df = pd.read_csv(SAMPLE_METADATA, index_col=0, dtype='category')

In [11]:
sample_metadata_df.head()

,Author_Sex_harmonized,Strain,Diet,Treatment,Age,Mean_age_in_weeks,IsControl
Sample,,,,,,,
SC1-01_SAMN18754353,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,Y
SC1-01_SAMN18754354,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,Y
SC1-01_SAMN18754356,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,Y
SC1-01_SAMN18754357,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,Y
SC1-01_SAMN18754358,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,Y


In [12]:

sample_metadata_df['IsControl'] = (sample_metadata_df['IsControl'] == 'Y')

obs_df = pd.merge(obs_df, sample_metadata_df, left_on="Sample", right_index=True, how="left")

In [13]:
obs_df.head()

,CellID,Dataset,Sample,total_counts,n_genes,pct_mito,Author,Source,Data_source_link,GEO,Publication_Link,Suspension,Technology,Author_Sex_harmonized,Strain,Diet,Treatment,Age,Mean_age_in_weeks,IsControl
SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01,SC1-01_SAMN18754353,17459.0,5884,0.131737,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,True
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01,SC1-01_SAMN18754353,17067.0,5727,0.117185,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,True
SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01,SC1-01_SAMN18754353,5183.0,2578,0.501640,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,True
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,SC1-01,SC1-01_SAMN18754353,11905.0,4965,0.201596,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,True
SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01_SAMN18754353_AAACGAACACCCTTAC,SC1-01,SC1-01_SAMN18754353,2001.0,1125,2.598701,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3,NaN,C57BL/6J,Normal chow,Ad lib. NC,NaN,NaN,True


### Let's add the published cell type annotations 


In [14]:
known_labels_df = pd.read_csv(KNOWN_LABELS, index_col=0, dtype='object')

known_labels_df = known_labels_df.astype('category')

known_labels_df.head()


,cluster_id_label,supertype_id_label,subclass_id_label,class_id_label,C66_named,C185_named,C286_named,C465_named
CellID,,,,,,,,
SC1-01_SAMN18754353_AAACCCACAACCAGAG,NaN,NaN,NaN,NaN,C66-12: Gpr149.GLU-3,C185-30: Hmcn2.Gpr149.GLU-3,C286-45: Rai14.Hmcn2.Gpr149.GLU-3,C465-78: Megf11.Rai14.Hmcn2.Gpr149.GLU-3
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,NaN,NaN,NaN,NaN,C66-10: Fezf1.GLU-3,C185-23: Arhgap36.Fezf1.GLU-3,C286-32: Gpr88.Arhgap36.Fezf1.GLU-3,C465-55: Npffr1.Gpr88.Arhgap36.Fezf1.GLU-3
SC1-01_SAMN18754353_AAACCCAGTGAGATTA,NaN,NaN,NaN,NaN,C66-53: Etnppl.Astrocytes,C185-139: Itih3.Etnppl.Astrocytes,C286-215: Serpina3n.Itih3.Etnppl.Astrocytes,C465-351: C4b.Serpina3n.Itih3.Etnppl.Astrocytes
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,NaN,NaN,NaN,NaN,C66-10: Fezf1.GLU-3,C185-22: Unassigned.Fezf1.GLU-3,C286-35: Sox14.Unassigned.Fezf1.GLU-3,C465-59: Sox14.Unassigned.Fezf1.GLU-3
SC1-01_SAMN18754353_AAACGAACACCCTTAC,NaN,NaN,NaN,NaN,C66-30: Onecut2.GABA-2,C185-74: Onecut3.Onecut2.GABA-2,C286-116: Shisal2b.Onecut3.Onecut2.GABA-2,C465-193: Adarb2.Shisal2b.Onecut3.Onecut2.GABA-2


In [15]:
obs_df = pd.merge(obs_df, known_labels_df, left_index=True, right_index=True, how="left")

### Let's export the updates

In [16]:
assert not any(obs_df['IsControl'].isna()), 'IsControl variable incomplete -- triple check please'
assert not any(obs_df['Sample'].isna()), 'Sample variable incomplete -- triple check please'

obs_df.head(3)

,CellID,Dataset,Sample,total_counts,n_genes,pct_mito,Author,Source,Data_source_link,GEO,...,Mean_age_in_weeks,IsControl,cluster_id_label,supertype_id_label,subclass_id_label,class_id_label,C66_named,C185_named,C286_named,C465_named
SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01_SAMN18754353_AAACCCACAACCAGAG,SC1-01,SC1-01_SAMN18754353,17459.0,5884,0.131737,Affinati,SRA,NaN,GSE172204,...,NaN,True,NaN,NaN,NaN,NaN,C66-12: Gpr149.GLU-3,C185-30: Hmcn2.Gpr149.GLU-3,C286-45: Rai14.Hmcn2.Gpr149.GLU-3,C465-78: Megf11.Rai14.Hmcn2.Gpr149.GLU-3
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01_SAMN18754353_AAACCCAGTCATCTAG,SC1-01,SC1-01_SAMN18754353,17067.0,5727,0.117185,Affinati,SRA,NaN,GSE172204,...,NaN,True,NaN,NaN,NaN,NaN,C66-10: Fezf1.GLU-3,C185-23: Arhgap36.Fezf1.GLU-3,C286-32: Gpr88.Arhgap36.Fezf1.GLU-3,C465-55: Npffr1.Gpr88.Arhgap36.Fezf1.GLU-3
SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01_SAMN18754353_AAACCCAGTGAGATTA,SC1-01,SC1-01_SAMN18754353,5183.0,2578,0.501640,Affinati,SRA,NaN,GSE172204,...,NaN,True,NaN,NaN,NaN,NaN,C66-53: Etnppl.Astrocytes,C185-139: Itih3.Etnppl.Astrocytes,C286-215: Serpina3n.Itih3.Etnppl.Astrocytes,C465-351: C4b.Serpina3n.Itih3.Etnppl.Astrocytes


In [17]:
write_elem(h5ad, 'obs', obs_df)

h5ad.close()

### validate

In [18]:
import scanpy as sc

sc.read_h5ad(annconcat_fp, backed='r')

AnnData object with n_obs × n_vars = 2030770 × 56939 backed at '/REDACTED-path/HypoMap3D-data/v0.9/2b_clean/annconcat.h5ad'
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene'